[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/soccer-vision/blob/master/examples/finetune_pitch.ipynb)

# Pitch-keypoint fine-tune (Phase 3.5b)
Trains YOLOv8-pose on the 21-point youth landmark schema. Output:
runs/pose/pitch_v0/weights/best.pt -> publish as the pitch-v1 release asset.
kpt_shape and flip_idx MUST match soccer_vision.pitch.landmarks.

In [ ]:
!pip install -q "ultralytics>=8.2" roboflow
from pathlib import Path
WORK = Path("/content/work"); WORK.mkdir(exist_ok=True)
%cd /content/work


## Option A — Roboflow dataset
Only if your dataset lives in Roboflow. Otherwise skip to Option B below.


In [ ]:
import os
from google.colab import userdata
os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
# TODO when running: replace workspace/project/version
project = rf.workspace("YOUR_WORKSPACE").project("pitch_v1")
dataset = project.version(1).download("yolov8")
print("Dataset at:", dataset.location)


## Option B — local dataset (no Roboflow)
If you built the dataset with `python -m soccer_vision.dataset_export`, upload
`dataset.zip` to Colab (or copy from Drive) and run the next cell INSTEAD of the
Roboflow download above. The zip's data.yaml already carries kpt_shape and flip_idx.

In [ ]:
import zipfile
from pathlib import Path
from types import SimpleNamespace

ZIP = Path("/content/dataset.zip")
DRIVE_ZIP = Path("/content/drive/MyDrive/soccer-vision/dataset.zip")
if not ZIP.exists():
    from google.colab import drive
    drive.mount("/content/drive")
    if DRIVE_ZIP.exists():
        !cp "{DRIVE_ZIP}" /content/dataset.zip
assert ZIP.exists(), "upload dataset.zip or place it in Drive soccer-vision/"
target = Path("/content/dataset")
with zipfile.ZipFile(ZIP) as zf:
    zf.extractall(target)
dataset = SimpleNamespace(location=str(target))
print("Using local dataset at", dataset.location)


IMPORTANT: data.yaml must declare kpt_shape: [21, 3] and
flip_idx: [1,0,3,2,5,4,6,7,8,10,9,12,11,14,13,16,15,18,17,20,19]
(matches soccer_vision.pitch.landmarks.FLIP_IDX). Without flip_idx, fliplr
augmentation corrupts left/right keypoints. The next cell patches the exported
data.yaml in case Roboflow omits it.

In [ ]:
import yaml
dy = Path(dataset.location) / "data.yaml"
cfg = yaml.safe_load(dy.read_text())
cfg["kpt_shape"] = [21, 3]
cfg["flip_idx"] = [1, 0, 3, 2, 5, 4, 6, 7, 8, 10, 9, 12, 11, 14, 13, 16, 15, 18, 17, 20, 19]
dy.write_text(yaml.safe_dump(cfg))
print(cfg)

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8s-pose.pt")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100, imgsz=1280, batch=8, patience=20,
    mosaic=1.0, mixup=0.15, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    scale=0.5, fliplr=0.5,
    project=".", name="pitch_v0",
)

In [ ]:
metrics = model.val()
print("pose mAP50:", metrics.pose.map50)
from google.colab import files
files.download("/content/work/pitch_v0/weights/best.pt")

## Optional — continue from a previous run (warm start)
Squeeze more from a model that was still improving when it early-stopped. Needs
the dataset present (run Option B first if this is a fresh runtime) and `last.pt`
uploaded to /content or Drive `soccer-vision/last.pt`.


In [ ]:
from ultralytics import YOLO
from pathlib import Path
WARM = Path("/content/last.pt")
if not WARM.exists():
    from google.colab import drive; drive.mount("/content/drive")
    !cp "/content/drive/MyDrive/soccer-vision/last.pt" /content/last.pt
loc = dataset.location if "dataset" in dir() else "/content/dataset"
model = YOLO("/content/last.pt")          # warm-start from the prior run's e42
results = model.train(
    data=f"{loc}/data.yaml",
    epochs=60, imgsz=1280, batch=8,
    patience=40,        # generous: the box-fitness plateau must not cut the rising pose curve
    close_mosaic=15,    # mosaic off for the last 15 epochs -> usual late metric bump
    name="pitch_v1_cont",
)
# a brief dip during warmup (LR re-ramps) is normal; it should then pass the prior 0.35
